# Week 6 · Sunday quiz

**Rules.** Part A: write the exact output of each snippet on paper, then run it. Part B: on paper, then open the answer. Part C: write each function on paper, then type it and run the check. About 40 minutes.

Log all three scores, and for each miss write **which idea** it was (e.g. "A2: stride in the size formula").

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import checks; checks.load(globals(), "07")      # defines check(n)

## Part A · Predict the output

**A1**

In [ ]:
import numpy as np
import torch
import torch.nn as nn
print(nn.Conv2d(3, 8, 3)(torch.zeros(1, 3, 10, 10)).shape)

**A2**

In [ ]:
print(nn.Conv2d(3, 8, 3, padding=1, stride=2)(torch.zeros(1, 3, 10, 10)).shape)

**A3**

In [ ]:
print(sum(p.numel() for p in nn.Conv2d(4, 2, 3).parameters()))

**A4**

In [ ]:
print(nn.MaxPool2d(2)(torch.arange(16.0).reshape(1, 1, 4, 4)))

**A5**

In [ ]:
print(nn.Flatten()(torch.zeros(5, 4, 3, 3)).shape)

**A6**

In [ ]:
d = nn.Dropout(0.3)
d.eval()
print(d(torch.ones(3)))

**A7**

In [ ]:
bn = nn.BatchNorm1d(2)
print(bn(torch.tensor([[1.0, 10.0], [3.0, 30.0]])).detach())

**A8**

In [ ]:
w = torch.tensor([1.0], requires_grad=True)
opt = torch.optim.SGD([w], lr=0.1, weight_decay=1.0)
(0 * w).sum().backward()
opt.step()
print(w.detach())

**A9**

In [ ]:
g = torch.zeros(2, requires_grad=True)
(6 * g[0] + 8 * g[1]).backward()
print(torch.nn.utils.clip_grad_norm_([g], max_norm=5.0), g.grad)

**A10**

In [ ]:
x = np.arange(9.0).reshape(3, 3); K = np.ones((2, 2))
print([[float(np.sum(x[i:i+2, j:j+2] * K)) for j in range(2)] for i in range(2)])

## Part B · On paper

**B1.** An input `(N, 1, 28, 28)` goes through `Conv2d(1, 6, 5)` (no padding), `MaxPool2d(2)`, `Conv2d(6, 16, 5)`, `MaxPool2d(2)`, `Flatten`, `Linear(?, 10)`. Give every shape, the `?`, and the total number of parameters.

<details><summary>Answer</summary>

$28 - 5 + 1 = 24$: `(N, 6, 24, 24)`; pool `(N, 6, 12, 12)`; $12 - 5 + 1 = 8$: `(N, 16, 8, 8)`; pool `(N, 16, 4, 4)`; flatten $16 \cdot 4 \cdot 4 = 256$, so `Linear(256, 10)`. Parameters: $6 \cdot 1 \cdot 25 + 6 = 156$; $16 \cdot 6 \cdot 25 + 16 = 2416$; $256 \cdot 10 + 10 = 2570$; total $5142$. (This is LeNet-5's front half.)

</details>

**B2.** Batch norm on one feature with the batch values $(1, 3, 5, 7)$, $\gamma = 1$, $\beta = 0$ (ignore $\varepsilon$). The outputs? Then the same layer gets a batch of one sample in training mode: what goes wrong?

<details><summary>Answer</summary>

$\mu = 4$, $\sigma^2 = \frac{9 + 1 + 1 + 9}{4} = 5$, $\sigma = 2.236$: outputs $(-1.342, -0.447, 0.447, 1.342)$. With one sample the batch variance is 0 and the value minus the mean is 0, so there's nothing to normalize; PyTorch raises an error in training mode. Use `eval()` (running averages) or layer norm.

</details>

**B3.** Dropout with $p = 0.5$. A unit's value is $h = 3$. What values can its output take during training, with what probabilities, and what is the expected output? What is the output at test time?

<details><summary>Answer</summary>

$6$ (kept, $3 / 0.5$) with probability $0.5$, or $0$ with probability $0.5$; expected value $0.5 \cdot 6 = 3 = h$. At test time, $3$: dropout is off and no scaling is needed (that's what the $1/(1-p)$ during training is for).

</details>

## Part C · Checked functions

Write each function on paper first, then type it and run the check.

**C1 · A convolution by hand, from memory.** One channel, stride 1, no padding: `out[i, j] = np.sum(x[i:i+a, j:j+b] * K)` for every position where the kernel fits. Return the `(h - a + 1, w - b + 1)` array.

**Example**: what goes in, and what should come out:
```python
conv2d_single(np.arange(9.0).reshape(3, 3), np.ones((2, 2)))    # → [[ 8. 12.]
                                                               #    [20. 24.]]
```

In [ ]:
def conv2d_single(x, K):
    pass  # your code

In [ ]:
check(1)

**C2 · Batch norm (training mode), from memory.** Each column standardized with its batch mean and variance (`axis=0`), then times `gamma` plus `beta`.

**Example**: what goes in, and what should come out:
```python
batchnorm_train(np.array([[1.0], [3.0], [5.0], [7.0]]), np.ones(1), np.zeros(1)).ravel().round(3)    # → [-1.342 -0.447  0.447  1.342]
```

In [ ]:
def batchnorm_train(X, gamma, beta, eps=1e-5):
    pass  # your code

In [ ]:
check(2)

**C3 · Early stopping, from memory.** Return `(stop_epoch, best_epoch)` for a list of validation losses and a patience, as in `02` §4.

**Example**: what goes in, and what should come out:
```python
early_stop([1.0, 0.8, 0.75, 0.77, 0.74, 0.76, 0.78, 0.79], 2)    # → (6, 4)   06's I8
```

In [ ]:
def early_stop(val_losses, patience):
    pass  # your code

In [ ]:
check(3)

**C4 · Booklet 7: one k-means step.** Assign every row of `X` `(n, d)` to its nearest centre in `mu` `(k, d)` (Euclidean distance), then move each centre to the mean of its points. Return `(c, new_mu)`: the labels `(n,)` and the new centres `(k, d)`. (The check never leaves a centre without points.)

**Example**: what goes in, and what should come out:
```python
kmeans_step(np.array([[0.0], [1.0], [9.0], [10.0]]), np.array([[0.0], [5.0]]))    # → (array([0, 0, 1, 1]), array([[0.5], [9.5]]))
```

In [ ]:
def kmeans_step(X, mu):
    pass  # your code

In [ ]:
check(4)

**C5 · Booklet 6: PCA scores.** Centre `X`; covariance $S = X_c^\top X_c / (n - 1)$; `np.linalg.eigh`; sort the eigenvectors by eigenvalue, largest first; return $Z = X_c W_{:, :k}$, **shape `(n, k)`** (the check allows each column's sign to flip).

**Example**: what goes in, and what should come out:
```python
pca_scores(np.array([[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]]), 1).ravel().round(3)    # → [-1.414  0.     1.414]   (or the signs flipped)
```

In [ ]:
def pca_scores(X, k):
    pass  # your code

In [ ]:
check(5)

---
Log all three scores (A, B, C) and every miss **by topic**, in the tracker. Week 7's recaps are built from them.